In [1]:
#Yeni bir veri seti ile eğitim yapacağız daha yuksek bir YOLO modeli ile 
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.3 MB/s eta 0:00:0000:01


In [2]:
import os

path = '/kaggle/input'
print(os.listdir(path))

for item in os.listdir(path):
    full = os.path.join(path, item)
    if os.path.isdir(full):
        print(f"\n{item}/", os.listdir(full)[:10])

['datasets']

datasets/ ['luiscrmartins']


In [3]:
path = '/kaggle/input/datasets/luiscrmartins/surveillance-images-for-person-detection'
print(os.listdir(path))
path = '/kaggle/input/datasets/luiscrmartins/surveillance-images-for-person-detection/person-3'
print(os.listdir(path))

['person-3']
['README.dataset.txt', 'README.roboflow.txt', 'data.yaml', 'train']


In [7]:
import os, shutil, random

dataset_path = '/kaggle/input/datasets/luiscrmartins/surveillance-images-for-person-detection/person-3/train'
output_path = '/kaggle/working/dataset'
# eğitimde kullanılacak train val ve test klasörleri oluşturuldu
for split in ['train', 'val', 'test']:
    os.makedirs(f'{output_path}/{split}/images', exist_ok=True)
    os.makedirs(f'{output_path}/{split}/labels', exist_ok=True)

images = [f for f in os.listdir(f'{dataset_path}/images') if f.endswith('.jpg')]
random.seed(42)
random.shuffle(images)

# veri setinin %70 i eğitim, %20 si validasyon %10 u da test olarak ayırdık.
train_end = int(len(images) * 0.7)
val_end = int(len(images) * 0.9)

splits = {
    'train': images[:train_end],
    'val': images[train_end:val_end],
    'test': images[val_end:]
}

for split, file_list in splits.items():
    for img in file_list:
        label = img.replace('.jpg', '.txt')
        shutil.copy(f'{dataset_path}/images/{img}', f'{output_path}/{split}/images/{img}')
        shutil.copy(f'{dataset_path}/labels/{label}', f'{output_path}/{split}/labels/{label}')

for split in ['train', 'val', 'test']:
    count = len(os.listdir(f'{output_path}/{split}/images'))
    print(f'{split}: {count} görüntü')

train: 4622 görüntü
val: 1320 görüntü
test: 661 görüntü


In [9]:
yaml_content = """
train: /kaggle/working/dataset/train/images
val: /kaggle/working/dataset/val/images
test: /kaggle/working/dataset/test/images

nc: 1
names: ['person']
"""

with open('/kaggle/working/dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print("data.yaml oluşturuldu")

# Kontrol et
with open('/kaggle/working/dataset/data.yaml', 'r') as f:
    print(f.read())

data.yaml oluşturuldu

train: /kaggle/working/dataset/train/images
val: /kaggle/working/dataset/val/images
test: /kaggle/working/dataset/test/images

nc: 1
names: ['person']



In [10]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # medium boyut

results = model.train(
    data='/kaggle/working/dataset/data.yaml',
    epochs=20,
    imgsz=640,
    batch=16,
    patience=15,
    name='surveillance_v1',
    
    # Augmentation
    mosaic=1.0,
    mixup=0.15,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    
    # Optimization
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    cos_lr=True,  # cosine learning rate scheduler
)

Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=surveillance_v1-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam

In [13]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/runs/detect/surveillance_v1-4/weights/best.pt')

results = model.val(data='/kaggle/working/dataset/data.yaml', split='test')

Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1980.6±240.8 MB/s, size: 185.0 KB)
val: Scanning /kaggle/working/dataset/test/labels... 661 images, 144 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 661/661 1.4Kit/s 0.5s0.1s
val: New cache created: /kaggle/working/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 42/42 3.2it/s 13.1s0.3s
                   all        661       1302      0.912      0.917      0.959      0.708
Speed: 0.7ms preprocess, 15.8ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-3
